# Group-Aware Classification of Mouse Protein Expression

## Objective

This notebook develops a baseline machine-learning model for predicting the eight experimental classes from 77 protein-expression measurements.

Because each biological mouse contributes 15 repeated measurements, all measurements from the same mouse must remain together during model evaluation. This prevents the model from being tested on mice it has already encountered during training.

Missing-value imputation, feature scaling, and model fitting will be performed using training data only.

## 1. Load the modelling data

The dataset is loaded again so this notebook can run independently. Only protein-expression measurements are used as predictors. The experimental class is the target, while the biological mouse identifier is retained exclusively for group-aware evaluation.

In [2]:
from ucimlrepo import fetch_ucirepo

mice = fetch_ucirepo(id=342)

X = mice.data.features.drop(
    columns=["Genotype", "Treatment", "Behavior"]
)

y = mice.data.targets["class"]

groups = (
    mice.data.ids["MouseID"]
    .str.split("_")
    .str[0]
)

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)
print("Biological mice:", groups.nunique())

Predictor shape: (1080, 77)
Target shape: (1080,)
Biological mice: 72


## 2. Create a group-aware evaluation strategy

Three-fold stratified group cross-validation is used.

- **Group-aware:** all 15 measurements from one mouse remain together.
- **Stratified:** each fold should contain a similar distribution of the eight classes.
- **Cross-validation:** the model will eventually be evaluated three times using different test mice.
- **The number of folds depends mainly on:**
 1. Number of independent biological subjects.
 2. Number of subjects in the smallest class.
 3. Having enough mice for both training and reliable testing.

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

group_splitter = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

first_split = next(
    group_splitter.split(X, y, groups=groups)
)

train_indices, test_indices = first_split

train_mice = set(groups.iloc[train_indices])
test_mice = set(groups.iloc[test_indices])

print("Training measurements:", len(train_indices))
print("Testing measurements:", len(test_indices))

print("Training mice:", len(train_mice))
print("Testing mice:", len(test_mice))

print("Mice present in both sets:", train_mice.intersection(test_mice))

Training measurements: 720
Testing measurements: 360
Training mice: 48
Testing mice: 24
Mice present in both sets: set()


## 3. Define the preprocessing steps

Missing protein measurements will be replaced with the median value of the corresponding protein. For every cross-validation fold, the median must be calculated using the training mice only and then applied to both the training and testing measurements.

In [7]:
from sklearn.impute import SimpleImputer

median_imputer = SimpleImputer(
    strategy="median"
)

### Standardize protein measurements

The protein variables have different numerical ranges. Standardization places them on comparable scales so that proteins with larger numerical values do not influence the model simply because of their measurement scale.

For every fold, the mean and standard deviation will be learned from the training mice only and then applied to both training and testing measurements.

In [8]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()

### Define the baseline classifier

Logistic regression will be used as the first baseline model. It predicts one of the eight experimental classes using the standardized protein measurements.

In [10]:
from sklearn.linear_model import LogisticRegression

baseline_classifier = LogisticRegression(
    max_iter=5000,
    class_weight="balanced"
)

### Build the modelling pipeline

The pipeline performs missing-value imputation, standardization, and classification in the correct order. During cross-validation, all preprocessing rules are learned using only the training mice.

In [11]:
from sklearn.pipeline import Pipeline

baseline_pipeline = Pipeline(
    steps=[
        ("imputer", median_imputer),
        ("scaler", standard_scaler),
        ("classifier", baseline_classifier)
    ]
)

baseline_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto accou

## 4. Evaluate the baseline model

The pipeline is evaluated using three-fold stratified group cross-validation. Balanced accuracy and macro F1 are used because the experimental classes contain different numbers of mice.

In [13]:
from sklearn.model_selection import cross_validate

scoring_metrics = {
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro"
}

cv_results = cross_validate(
    estimator=baseline_pipeline,
    X=X,
    y=y,
    groups=groups,
    cv=group_splitter,
    scoring=scoring_metrics
)

cv_results

{'fit_time': array([0.03069258, 0.01572752, 0.03070211]),
 'score_time': array([0., 0., 0.]),
 'test_balanced_accuracy': array([0.60972222, 0.65833333, 0.67916667]),
 'test_macro_f1': array([0.61188292, 0.65653825, 0.661005  ])}